[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/xpbd/xpbd.ipynb)

# A Chain of Rigid Links, by Position-Based Dynamics

This example lets a chain of ten rigid links, each a solid ellipsoid with three different axes, hang from a pin and fall from a horizontal arc, held together at its joints. Being extended bodies, the links do not only move but turn and twist about their own axes as the chain whips around. Each step lets every link move freely, then pulls the joints closed again: wherever two links have drifted apart, both are nudged back along the line between their anchors, each by as much as its inertia allows. This is extended position-based dynamics, the method of game physics, and it keeps the joints closed to a tiny fraction of a link.

A link's pose is a motor and its rate a bivector; its inertia maps rates to forques, lines of force. The gap at a joint is the line joining the two anchors, their regressive product: its norm is how far apart they are, its direction the line the correcting load acts along. Pulled into a link's frame, that forque turns through the inverse inertia into the twist it would cause, and the twist paired with the forque, `twist & forque`, is the link's compliance along the line. One multiplier closes the gap, and each link moves by the exponential of its twist.

In matrix notation the projection reads as explicit Jacobian matrices, a generalized inverse-mass matrix built from them and the inverse mass matrix, a solve for Lagrange multipliers, and separate linear and angular impulses on velocities and quaternion poses.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import tee
from typing import NamedTuple

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import show_animation
from examples.mechanics.xpbd import render

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Point = ga.gatype.antivector()                                                 # Antivector-
Motor = ga.gatype.rotor()                                                      # Rotor+
# A link's rate of motion, a bivector, and a load along a line, a forque.
Rate = ga.gatype.bivector()                                                    # Bivector+
Forque = ga.gatype.antibivector()                                              # Antibivector-
Inertia = ga.gatype((Forque, Rate))
InverseInertia = ga.gatype((Rate, Forque))
origin = mv.w.dual()                                                           # [] Point-


class Chain(NamedTuple):
    """The links' state, batched over the bodies: the pin first, then the links."""
    motor: Motor                  # [bodies] Motor+
    # The rate in the body's frame.
    rate: Rate                    # [bodies] Rate+
    # The mass-weighted centre, in the body's frame.
    first_moment: Point           # [bodies] Point-
    inertia: Inertia              # [bodies] Forque- <- Rate+
    # Zero for the pin, which nothing moves.
    inverse_inertia: InverseInertia   # [bodies] Rate+ <- Forque-
    damping: float
    # A point at infinity: the gravitational acceleration.
    gravity: Point                # [] Point-


class Joints(NamedTuple):
    """Joints whose links do not overlap, relaxed together: the two bodies of each, its anchor on each, in that
    body's frame, and its compliance."""
    bodies: np.ndarray            # [ends, joints] indices
    anchors: Point                # [ends, joints] Point-
    compliance: float

## 1. The chain

Each link is six unit masses on its axes, placed to give it the inertia of a solid ellipsoid, so its three moments of inertia differ. Neighbours are joined at their ends; the chain starts on a horizontal arc, each link its neighbour carried along x and turned about z, the powers of one motor. The pin is a body of its own with zero inverse inertia, an infinite mass, so nothing moves it, and the first link swings from it. The joints are split into two halves whose links do not overlap, so that each half relaxes in one batch.

In [ ]:
# The links, the spacing of their joints, the semi-axes of each link's ellipsoid, and the turn between
# neighbours along the starting arc; the joints' compliance, the damping and gravity.
n_links, spacing, curl = 10, 9e-2, 0.3
semi_axes = np.array([0.62, 0.38, 0.22]) * spacing
compliance, damping, gravity = 1e-9, 1e-3, 0.5

In [ ]:
def chain(n_links: int, spacing: float, semi_axes: np.ndarray, curl: float, compliance: float, damping: float,
          gravity: float) -> tuple[Chain, list[Joints]]:
    """A chain of solid ellipsoidal links hanging from a pin at the origin, in gravity along -z, and its joints in two
    halves whose links do not overlap: the pin and the first link, the second and the third, and so on, then the
    others."""
    n_bodies = n_links + 1
    # Six unit masses a link, out along its axes by the semi-axes times the root of three fifths: together they have a
    # solid ellipsoid's inertia. Each mass point joined with its own motion under a rate is its momentum.
    offsets = mv.direction(np.eye(3)) * (semi_axes * np.sqrt(3 / 5))           # [3] Point-
    mass_points = concatenate([origin + offsets, origin - offsets])            # [points] Point-
    inertia = (mass_points & mass_points.commutator(Rate)).sum(axis=-1)        # [] Forque- <- Rate+
    # The pin has an infinite mass: no inverse inertia.
    mobility = np.concatenate([[0.0], np.ones(n_links)])                       # [bodies] float
    # From one body to the next along the arc: carried half the spacing along x, turned about z, carried on.
    half_shift = (mv.xw * (spacing / 4)).exp()                                 # [] Motor+
    neighbour = half_shift * (mv.xy * (curl / 2)).exp() * half_shift           # [] Motor+
    state = Chain(
        motor=(neighbour.log() * np.arange(n_bodies)).exp().cast(Motor.output_subspace),   # [bodies] Motor+
        rate=mv.bivector().broadcast_to(n_bodies),
        first_moment=mass_points.sum(axis=-1)[None].broadcast_to(n_bodies),
        inertia=inertia[None].broadcast_to(n_bodies),
        inverse_inertia=inertia.inverse()[None].broadcast_to(n_bodies) * mobility,
        damping=damping,
        gravity=mv.direction([0.0, 0.0, -gravity]),
    )
    # Joint j joins the +x end of body j to the -x end of body j + 1.
    link = np.arange(n_links)
    pairs = np.array([link, link + 1])                                         # [ends, joints]
    anchors = origin + mv.direction([1.0, 0.0, 0.0]) * (np.array([[0.5], [-0.5]]) * spacing * np.ones(n_links))   # [ends, joints] Point-
    halves = [Joints(pairs[:, half], anchors[:, half], compliance) for half in (slice(0, None, 2), slice(1, None, 2))]
    return state, halves


initial, halves = chain(n_links, spacing, semi_axes, curl, compliance, damping, gravity)

## 2. One step

A step integrates every link freely by a Lie-group step under gravity and damping, then closes each half of the joints in turn, reads the rates back from the motors' change, and finally cancels the joints' relative velocities with impulses along them.

In [ ]:
def free_step(chain: Chain, dt: float) -> tuple[Motor, Rate]:
    """One Runge-Kutta step of every link's rate under its weight and damping less the gyroscopic turn of its momentum,
    then its motor along the stepped rate."""
    def acceleration(rate: Rate) -> Rate:
        # The weight is the line through the centre of mass along gravity, in the body's frame; damping opposes the rate.
        weight = chain.first_moment & (chain.motor << chain.gravity)
        loads = weight - (rate * chain.damping).dual()
        return chain.inverse_inertia(loads - chain.inertia(rate).commutator(rate))

    k1 = acceleration(chain.rate)
    k2 = acceleration(chain.rate + k1 * (dt / 2))
    k3 = acceleration(chain.rate + k2 * (dt / 2))
    k4 = acceleration(chain.rate + k3 * dt)
    rate = chain.rate + (k1 + 2 * k2 + 2 * k3 + k4) * (dt / 6)
    return chain.motor * (rate * (-dt / 2)).exp(), rate


def closed(motors: Motor, anchors: Point, inverse_inertia: InverseInertia, compliance: float, dt: float) -> Motor:
    """The two bodies of each joint `[ends, joints]` moved to close the gap between its anchors."""
    world_anchors = motors >> anchors                                          # [ends, joints] Point-
    # The join of the two anchors: its norm is how far apart they are, its direction the line the correcting load
    # acts along. A closed joint's line vanishes; the small offset keeps its direction and multiplier at zero.
    line = world_anchors[0] & world_anchors[1]                                 # [joints] Forque-
    gap = line.norm()
    # Pulled into each body's frame, the load turns into the twist it would cause; paired with it, the body's
    # compliance along the line.
    body = motors << line / (gap + 1e-24)                                      # [ends, joints] Forque-
    twists = inverse_inertia(body)                                             # [ends, joints] Rate+
    total_compliance = (twists & body).sum(axis=0) + compliance / dt**2 + 1e-24   # [joints] Scalar+
    # One multiplier closes the gap, the two bodies moved by equal and opposite twists.
    impulse = twists * (gap / total_compliance * np.array([[1.0], [-1.0]]))    # [ends, joints] Rate+
    return motors * (impulse * -0.5).exp()


def stopped(motors: Motor, rates: Rate, anchors: Point, inverse_inertia: InverseInertia) -> Rate:
    """The rates of the two bodies of each joint `[ends, joints]` with the anchors' relative motion cancelled by an
    impulse along it."""
    # Each anchor joined with its own motion under its body's rate: its momentum line, in the world.
    moving = motors >> (anchors & anchors.commutator(rates))                   # [ends, joints] Forque-
    relative = moving[1] - moving[0]                                           # [joints] Forque-
    size = relative.scalar_norm_squared().square_root()                        # [joints] Scalar+
    body = motors << relative / (size + 1e-24)
    twists = inverse_inertia(body)
    total_compliance = (twists & body).sum(axis=0) + 1e-24
    return rates + twists * (size / total_compliance * np.array([[1.0], [-1.0]]))


def step(chain: Chain, halves: list[Joints], dt: float) -> Chain:
    """One step of the chain: a free step, the joints closed half by half, the rates read back from the motors'
    change, and the joints' relative motion cancelled half by half."""
    motor, _ = free_step(chain, dt)
    for joints in halves:
        pair = joints.bodies
        motor = motor.at[pair].set(closed(motor[pair], joints.anchors, chain.inverse_inertia[pair], joints.compliance, dt))
    motor = motor.normalized()
    rate = (~chain.motor * motor).log() * (-2 / dt)                            # [bodies] Rate+
    for joints in halves:
        pair = joints.bodies
        rate = rate.at[pair].set(stopped(motor[pair], rate[pair], joints.anchors, chain.inverse_inertia[pair]))
    return chain._replace(motor=motor, rate=rate)


def gaps(motor: Motor, halves: list[Joints]) -> Scalar:
    """How far apart the two anchors of every joint lie: the norm of their join."""
    def gap(joints: Joints) -> Scalar:
        world_anchors = motor[joints.bodies] >> joints.anchors
        return (world_anchors[0] & world_anchors[1]).norm()

    return concatenate([gap(joints) for joints in halves])

## 3. The swing

A hundred steps of four substeps each let the chain fall from its arc and swing. The projection is stable at any step; the substeps decide how closed the joints stay, here to a few millimetres, inside the links' overlapping ends. Each link is ray traced as its ellipsoid placed by its motor, its alternate octants darker so that its turning shows, over a floor that catches the shadows.

In [ ]:
n_steps, n_substeps, dt, frame_duration = 100, 4, 0.04, 60

In [ ]:
def swing(chain: Chain, n_steps: int, n_substeps: int, dt: float) -> Iterator[Chain]:
    """The chain at the start and after each step of `n_substeps` substeps."""
    yield chain
    for _ in range(n_steps):
        for _ in range(n_substeps):
            chain = step(chain, halves, dt / n_substeps)
        yield chain


shown, states = tee(swing(initial, n_steps, n_substeps, dt))
show_animation(render.animate_chain((state.motor for state in shown), semi_axes, spacing), "xpbd_chain", frame_duration)
motors = stack([state.motor for state in states])                              # [steps + 1, bodies] Motor+

In [ ]:
# checks
# The joints stay closed to within a tenth of the spacing, and the pin never leaves the origin.
joint_gaps = stack([gaps(motor, halves) for motor in motors])                    # [steps + 1, joints] Scalar+
assert joint_gaps.to_array().max() < 0.1 * spacing
assert ((motors[:, 0] >> origin) & origin).norm().to_array().max() < 1e-6